<a href="https://colab.research.google.com/github/Keistkmiya/Tugas2-MachineLearning/blob/main/Tugas2_Chapter3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Chapter 3: Data Wrangling

## Setup and Data Loading

In [1]:
import pandas as pd
import numpy as np

url_ev = "https://data.wa.gov/api/views/f6w7-q2d2/rows.csv?accessType=DOWNLOAD"
df_ev = pd.read_csv(url_ev)

print("Setup berhasil dan data telah dimuat!")
print(f"Total baris data: {len(df_ev)}")

Setup berhasil dan data telah dimuat!
Total baris data: 280833


## Creating New Features

Data wrangling sering kali melibatkan pembuatan fitur atau kolom baru dari informasi yang sudah ada. Dalam langkah ini, kita menghitung umur kendaraan dengan cara mengurangi tahun saat ini (2026) dengan tahun pembuatan mobil yang tertera pada kolom 'Model Year'.

In [2]:
current_year = 2026
df_ev['Vehicle_Age'] = current_year - df_ev['Model Year']

display(df_ev[['Make', 'Model', 'Model Year', 'Vehicle_Age']].head())

,Make,Model,Model Year,Vehicle_Age
0,NISSAN,LEAF,2013,13
1,TESLA,MODEL 3,2020,6
2,TESLA,MODEL 3,2018,8
3,FIAT,500E,2024,2
4,TESLA,MODEL Y,2020,6


## Grouping and Aggregating Data

Kita menggunakan teknik pengelompokan (*grouping*) untuk melihat ringkasan statistik dari dataset yang besar. Di sini, kita mengelompokkan data berdasarkan merek kendaraan ('Make') untuk mengetahui rata-rata jarak tempuh listrik ('Electric Range') yang ditawarkan oleh masing-masing produsen.

In [3]:
avg_range_by_make = df_ev.groupby('Make')['Electric Range'].mean().sort_values(ascending=False)

print("10 Merek dengan Rata-rata Jarak Tempuh Tertinggi:")
print(avg_range_by_make.head(10))

10 Merek dengan Rata-rata Jarak Tempuh Tertinggi:
Make
JAGUAR                  167.802632
WHEEGO ELECTRIC CARS    100.000000
TH!NK                   100.000000
CHEVROLET                76.944513
FIAT                     73.831169
NISSAN                   63.327741
SMART                    61.652174
AZURE DYNAMICS           56.000000
TESLA                    50.871151
LAND ROVER               50.685590
Name: Electric Range, dtype: float64


## Applying Custom Functions
Untuk logika yang lebih spesifik, kita menerapkan fungsi kustom menggunakan metode `apply`. Kita membuat fungsi untuk mengategorikan kendaraan ke dalam kelompok 'Long Range', 'Medium Range', atau 'Short Range' berdasarkan nilai pada kolom jarak tempuh listriknya.

In [4]:
def classify_range(r):
    if r > 200:
        return 'Long Range'
    elif r > 100:
        return 'Medium Range'
    else:
        return 'Short Range'

df_ev['Range_Category'] = df_ev['Electric Range'].apply(classify_range)

print("Hasil klasifikasi jarak tempuh:")
display(df_ev[['Make', 'Model', 'Electric Range', 'Range_Category']].head())

Hasil klasifikasi jarak tempuh:


,Make,Model,Electric Range,Range_Category
0,NISSAN,LEAF,75.0,Short Range
1,TESLA,MODEL 3,308.0,Long Range
2,TESLA,MODEL 3,215.0,Long Range
3,FIAT,500E,0.0,Short Range
4,TESLA,MODEL Y,291.0,Long Range


## Replacing Values

Sering kali kita menemukan nilai yang tidak konsisten atau ingin menyederhanakan label dalam dataset agar lebih mudah dianalisis. Kita bisa menggunakan metode `replace` untuk mengubah nilai tertentu secara massal. Misalnya, jika kita ingin menyeragamkan penulisan merek atau mengubah nilai tertentu menjadi label yang lebih deskriptif, teknik ini sangat efisien dibandingkan mengubahnya satu per satu secara manual.

In [5]:
df_ev['Make_Standardized'] = df_ev['Make'].replace('TESLA', 'Tesla Motors')

print("Perubahan label pada kolom 'Make':")
display(df_ev[['Make', 'Make_Standardized']].head())

Perubahan label pada kolom 'Make':


,Make,Make_Standardized
0,NISSAN,NISSAN
1,TESLA,Tesla Motors
2,TESLA,Tesla Motors
3,FIAT,FIAT
4,TESLA,Tesla Motors


## Renaming and Standardizing Columns

Untuk menjaga konsistensi dalam penulisan kode, kita sering kali perlu memastikan nama kolom seragam—misalnya dengan mengubah semuanya menjadi huruf kecil atau mengganti spasi dengan *underscore*. Dengan menggunakan metode `rename`, kita bisa memetakan nama kolom lama ke nama baru yang lebih ringkas. Hal ini sangat membantu kita saat melakukan pemanggilan kolom berkali-kali dalam proses analisis.

In [6]:
df_ev.columns = [col.lower().replace(' ', '_') for col in df_ev.columns]

print("Daftar nama kolom yang sudah kita standarisasi:")
print(df_ev.columns.tolist())

Daftar nama kolom yang sudah kita standarisasi:
['vin_(1-10)', 'county', 'city', 'state', 'postal_code', 'model_year', 'make', 'model', 'electric_vehicle_type', 'clean_alternative_fuel_vehicle_(cafv)_eligibility', 'electric_range', 'legislative_district', 'dol_vehicle_id', 'vehicle_location', 'electric_utility', '2020_census_tract', 'vehicle_age', 'range_category', 'make_standardized']


## Finding Minimum, Maximum, and Sum
Kita juga perlu sering melakukan pemeriksaan cepat terhadap nilai statistik dasar seperti nilai minimum, maksimum, atau total dari suatu kolom numerik. Pandas menyediakan fungsi bawaan seperti `min()`, `max()`, dan `sum()` yang bekerja sangat cepat bahkan pada jutaan baris data. Kita menggunakan informasi ini untuk memahami batasan data kita, misalnya tahun produksi tertua atau jangkauan listrik terjauh.

In [7]:
min_year = df_ev['model_year'].min()
max_range = df_ev['electric_range'].max()
total_cars = df_ev['vin_(1-10)'].count()

print(f"Mobil paling tua di dataset kita dibuat tahun: {min_year}")
print(f"Jarak tempuh terjauh yang ditemukan: {max_range} miles")
print(f"Total kendaraan dalam dataset: {total_cars}")

Mobil paling tua di dataset kita dibuat tahun: 1999
Jarak tempuh terjauh yang ditemukan: 337.0 miles
Total kendaraan dalam dataset: 280833


## Deleting Columns and Rows

Tidak semua data yang kita muat akan berguna untuk analisis atau pelatihan model. Terkadang, kita perlu membuang kolom yang mengandung informasi sensitif atau tidak relevan, serta menghapus baris tertentu yang tidak memenuhi kriteria kita. Kita menggunakan metode `drop` untuk menghapus kolom (dengan menentukan `axis=1`) atau menghapus baris (dengan menentukan `axis=0` atau berdasarkan indeks).

In [9]:
print("Daftar kolom saat ini:", df_ev.columns.tolist())
df_reduced = df_ev.drop(['dol_vehicle_id', 'base_msrp'], axis=1, errors='ignore')
df_reduced = df_reduced.drop(0, axis=0, errors='ignore')
df_modern = df_reduced[df_reduced['model_year'] >= 2010]

print(f"\nJumlah kolom setelah dihapus: {len(df_modern.columns)}")
print(f"Jumlah baris sekarang: {len(df_modern)}")

Daftar kolom saat ini: ['vin_(1-10)', 'county', 'city', 'state', 'postal_code', 'model_year', 'make', 'model', 'electric_vehicle_type', 'clean_alternative_fuel_vehicle_(cafv)_eligibility', 'electric_range', 'legislative_district', 'dol_vehicle_id', 'vehicle_location', 'electric_utility', '2020_census_tract', 'vehicle_age', 'range_category', 'make_standardized']

Jumlah kolom setelah dihapus: 18
Jumlah baris sekarang: 280802


## Handling Duplicates

Data duplikat dapat menyebabkan bias dalam analisis statistik yang kita lakukan. Kita menggunakan metode `duplicated()` untuk mengidentifikasi baris mana saja yang memiliki nilai yang sama persis dengan baris lainnya. Setelah itu, kita bisa menggunakan `drop_duplicates()` untuk menghapus baris-baris tersebut dan hanya menyisakan satu entri yang unik. Sesuai dengan standar publikasi O'Reilly yang kita ikuti, menjaga integritas data melalui penghapusan duplikat adalah langkah wajib dalam data wrangling.

In [10]:
duplicate_count = df_modern.duplicated().sum()
print(f"Jumlah baris yang duplikat: {duplicate_count}")

vin_duplicates = df_modern.duplicated(subset=['vin_(1-10)']).sum()
print(f"Jumlah VIN yang duplikat: {vin_duplicates}")

df_unique = df_modern.drop_duplicates()

print(f"\nJumlah baris setelah duplikat dihapus: {len(df_unique)}")

Jumlah baris yang duplikat: 24483
Jumlah VIN yang duplikat: 263541

Jumlah baris setelah duplikat dihapus: 256319


## Concatenating DataFrames

Ketika kita memiliki dua atau lebih DataFrame yang memiliki struktur kolom yang sama, kita bisa menggabungkannya secara vertikal (menumpuk baris) menggunakan fungsi `pd.concat`. Ini adalah teknik dasar yang sangat berguna jika kita bekerja dengan data yang terpisah secara waktu atau kategori. Di sini, kita akan mencoba membagi dataset kendaraan listrik kita menjadi dua bagian, lalu menyatukannya kembali untuk melihat bagaimana cara kerjanya.

In [11]:
df_1 = df_unique.head(5)
df_2 = df_unique.iloc[5:10]
df_combined = pd.concat([df_1, df_2], axis=0)

print(f"Jumlah baris df_1: {len(df_1)}")
print(f"Jumlah baris df_2: {len(df_2)}")
print(f"Jumlah baris setelah digabung: {len(df_combined)}")

display(df_combined)

Jumlah baris df_1: 5
Jumlah baris df_2: 5
Jumlah baris setelah digabung: 10


,vin_(1-10),county,city,state,postal_code,model_year,make,model,electric_vehicle_type,clean_alternative_fuel_vehicle_(cafv)_eligibility,electric_range,legislative_district,vehicle_location,electric_utility,2020_census_tract,vehicle_age,range_category,make_standardized
1,5YJ3E1EC8L,Kitsap,Bainbridge Island,WA,98110.0,2020,TESLA,MODEL 3,Battery Electric Vehicle (BEV),Clean Alternative Fuel Vehicle Eligible,308.0,23.0,POINT (-122.521 47.62759),PUGET SOUND ENERGY INC,5.303509e+10,6,Long Range,Tesla Motors
2,5YJ3E1EBXJ,King,Seattle,WA,98144.0,2018,TESLA,MODEL 3,Battery Electric Vehicle (BEV),Clean Alternative Fuel Vehicle Eligible,215.0,37.0,POINT (-122.30866 47.57874),CITY OF SEATTLE - (WA)|CITY OF TACOMA - (WA),5.303301e+10,8,Long Range,Tesla Motors
3,ZFAFFAC45R,Thurston,Yelm,WA,98597.0,2024,FIAT,500E,Battery Electric Vehicle (BEV),Eligibility unknown as battery range has not b...,0.0,2.0,POINT (-122.60735 46.94239),PUGET SOUND ENERGY INC,5.306701e+10,2,Short Range,FIAT
4,5YJYGDEE3L,King,Kent,WA,98030.0,2020,TESLA,MODEL Y,Battery Electric Vehicle (BEV),Clean Alternative Fuel Vehicle Eligible,291.0,47.0,POINT (-122.19975 47.37483),PUGET SOUND ENERGY INC||CITY OF TACOMA - (WA),5.303303e+10,6,Long Range,Tesla Motors
5,WBY1Z4C50F,Snohomish,Mukilteo,WA,98275.0,2015,BMW,I3,Plug-in Hybrid Electric Vehicle (PHEV),Clean Alternative Fuel Vehicle Eligible,72.0,21.0,POINT (-122.29196 47.89908),PUGET SOUND ENERGY INC,5.306104e+10,11,Short Range,BMW
6,5YJXCBE28G,King,Seattle,WA,98115.0,2016,TESLA,MODEL X,Battery Electric Vehicle (BEV),Clean Alternative Fuel Vehicle Eligible,200.0,43.0,POINT (-122.31676 47.68156),CITY OF SEATTLE - (WA)|CITY OF TACOMA - (WA),5.303300e+10,10,Medium Range,Tesla Motors
7,1G1RB6S51H,Island,Oak Harbor,WA,98277.0,2017,CHEVROLET,VOLT,Plug-in Hybrid Electric Vehicle (PHEV),Clean Alternative Fuel Vehicle Eligible,53.0,10.0,POINT (-122.64682 48.29077),PUGET SOUND ENERGY INC,5.302997e+10,9,Short Range,CHEVROLET
8,5YJ3E1EA7J,King,Kirkland,WA,98034.0,2018,TESLA,MODEL 3,Battery Electric Vehicle (BEV),Clean Alternative Fuel Vehicle Eligible,215.0,1.0,POINT (-122.22901 47.72201),PUGET SOUND ENERGY INC||CITY OF TACOMA - (WA),5.303302e+10,8,Long Range,Tesla Motors
9,5YJSA1E2XH,King,Seattle,WA,98107.0,2017,TESLA,MODEL S,Battery Electric Vehicle (BEV),Clean Alternative Fuel Vehicle Eligible,210.0,43.0,POINT (-122.38591 47.67597),CITY OF SEATTLE - (WA)|CITY OF TACOMA - (WA),5.303300e+10,9,Long Range,Tesla Motors
10,1G1FY6S02L,Yakima,Union Gap,WA,98903.0,2020,CHEVROLET,BOLT EV,Battery Electric Vehicle (BEV),Clean Alternative Fuel Vehicle Eligible,259.0,15.0,POINT (-120.71847 46.55029),PACIFICORP,5.307700e+10,6,Long Range,CHEVROLET


## Merging DataFrames

Dalam dunia nyata, informasi sering kali tersebar di beberapa tabel. Misalnya, kita punya tabel data kendaraan dan satu tabel lagi yang berisi informasi detail tentang produsen mobil tersebut. Kita menggunakan `pd.merge` untuk menyatukan informasi dari kedua tabel tersebut berdasarkan kolom yang sama (kunci). Teknik ini jauh lebih fleksibel daripada konkatenasi karena kita bisa memilih untuk melakukan *inner*, *outer*, *left*, atau *right join* sesuai kebutuhan analisis kita.

In [12]:
data_origin = {
    'make': ['TESLA', 'FORD', 'BMW', 'NISSAN', 'HYUNDAI'],
    'country': ['USA', 'USA', 'Germany', 'Japan', 'South Korea']
}
df_origin = pd.DataFrame(data_origin)
df_merged = pd.merge(df_unique, df_origin, on='make', how='left')

print("Hasil penggabungan tabel (Menambahkan kolom 'country'):")
display(df_merged[['make', 'model', 'country']].head(10))

print("\nJumlah kendaraan yang berhasil dipetakan negaranya:")
print(df_merged['country'].notnull().sum())

Hasil penggabungan tabel (Menambahkan kolom 'country'):


,make,model,country
0,TESLA,MODEL 3,USA
1,TESLA,MODEL 3,USA
2,FIAT,500E,NaN
3,TESLA,MODEL Y,USA
4,BMW,I3,Germany
5,TESLA,MODEL X,USA
6,CHEVROLET,VOLT,NaN
7,TESLA,MODEL 3,USA
8,TESLA,MODEL S,USA
9,CHEVROLET,BOLT EV,NaN



Jumlah kendaraan yang berhasil dipetakan negaranya:
149888


## Pivoting DataFrames

Pivoting adalah teknik tingkat lanjut untuk mengubah bentuk DataFrame agar lebih mudah dibaca dan dianalisis. Dengan fungsi `pivot_table()`, kita bisa mendefinisikan kolom mana yang akan menjadi indeks (baris), kolom mana yang akan menjadi *header* baru (kolom), dan nilai apa yang ingin kita agregasi (misalnya dihitung rata-ratanya). Hal ini sangat memudahkan kita untuk membandingkan statistik antar berbagai kategori dalam satu pandangan.

In [13]:
top_cities = df_merged['city'].value_counts().head(5).index
top_makes = df_merged['make'].value_counts().head(5).index

df_pivot_data = df_merged[df_merged['city'].isin(top_cities) & df_merged['make'].isin(top_makes)]
pivot_table = pd.pivot_table(
    df_pivot_data,
    values='electric_range',
    index='city',
    columns='make',
    aggfunc='mean'
)

print("Pivot Table: Rata-rata Jarak Tempuh (Miles) per Kota dan Merek")
display(pivot_table)

Pivot Table: Rata-rata Jarak Tempuh (Miles) per Kota dan Merek


make,CHEVROLET,FORD,KIA,NISSAN,TESLA
city,,,,,
Bellevue,56.493274,6.401235,27.834646,55.642361,51.381034
Bothell,64.147727,5.765823,25.002618,53.701124,44.255543
Redmond,70.376855,3.563467,24.205128,55.993852,53.694285
Seattle,88.132019,8.013966,31.577997,66.316106,57.048893
Vancouver,72.645885,9.687500,27.113176,74.529930,52.459534
